# Simple chunking implementation - Approach 1

In [1]:
def chunk_text(text, chunk_size=200):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks

In [2]:
text = """
Python is a programming language.
It is widely used for software development,
data analysis, machine learning and AI.
"""

chunks = chunk_text(text, 50)

for chunk in chunks:
    print("----")
    print(chunk)

----

Python is a programming language.
It is widely us
----
ed for software development,
data analysis, machin
----
e learning and AI.



# but the above simple implementation of chunks is not complete......so let's first split the text into words. - Approach 2

In [3]:
def chunk_text(text, chunk_size=10):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

In [4]:
text = """
Python is a programming language.
It is widely used for software development,
data analysis, machine learning and AI.
"""

chunks = chunk_text(text, 10)

for chunk in chunks:
    print("----")
    print(chunk)

----
Python is a programming language. It is widely used for
----
software development, data analysis, machine learning and AI.


# so There's no shared context. so lets use chunk overlap

In [5]:
def chunk_text(text, chunk_size=10, overlap=2):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        start = end - overlap

    return chunks

In [6]:
text = """
Python is a programming language.
It is widely used for software development,
data analysis, machine learning and AI.
"""
chunks = chunk_text(
    text,
    chunk_size=10,
    overlap=2
)
for chunk in chunks:
    print("----")
    print(chunk)

----
Python is a programming language. It is widely used for
----
used for software development, data analysis, machine learning and AI.
----
and AI.


# since this not a right content as there is no context awareness between chunks so lets use text-aware spiltter 

# There comes Langchain Text Splitter - approach 3

In [6]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

In [13]:
text = """
Python is a programming language.
It is widely used for software development,
data analysis, machine learning and AI.
"""

chunks = splitter.split_text(text)

for chunk in chunks:
    print("----")
    print(chunk)

----
Python is a programming language.
----
It is widely used for software development,
----
data analysis, machine learning and AI.


In [14]:
for i, chunk in enumerate(chunks):
    print(f"\nCHUNK {i}")
    print(chunk)


CHUNK 0
Python is a programming language.

CHUNK 1
It is widely used for software development,

CHUNK 2
data analysis, machine learning and AI.


# Load an embedding model

In [7]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
# embed chunks
embeddings = model.encode(chunks)
print(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[[-0.03509836  0.02055493 -0.02905505 ...  0.13021353  0.19174473
  -0.0240678 ]
 [-0.05535756  0.01999792 -0.0677697  ...  0.02073148  0.12268477
   0.0772597 ]
 [ 0.00380364 -0.00113125  0.03905817 ... -0.05188488 -0.03444219
  -0.06621905]]


In [17]:
print(embeddings.shape)

(3, 384)


In [18]:
print(embeddings[0])

[-3.50983553e-02  2.05549262e-02 -2.90550496e-02  1.25653464e-02
 -4.01899070e-02 -1.46645188e-01  5.25082182e-03  5.03757782e-02
 -4.63780686e-02 -2.02275719e-02 -3.43793966e-02  5.00154234e-02
  9.47559848e-02  1.24035478e-02  6.22891001e-02 -6.15294836e-02
 -7.65075311e-02 -1.99534725e-02  1.52971654e-03 -9.58153382e-02
 -5.93157392e-03  6.29708990e-02 -2.34328490e-02  2.24365555e-02
  2.57427823e-02 -1.16095776e-02 -1.07536148e-02 -9.67905112e-03
  2.45132558e-02  6.16728002e-03 -4.83584590e-02  1.01410627e-01
  4.12556641e-02  2.42749825e-02 -4.60887933e-03  5.20044193e-02
 -1.28395809e-02 -8.02411363e-02 -4.75864299e-02  5.49184391e-03
 -3.87001038e-02  3.87854427e-02 -6.10789694e-02 -3.17566171e-02
 -6.91966191e-02  5.25937825e-02 -3.75931971e-02 -1.86671950e-02
 -2.36246437e-02 -1.78272780e-02 -8.83034393e-02  1.01716593e-02
 -5.42289130e-02 -6.09493963e-02 -6.18806621e-03 -1.19047062e-02
  7.03026429e-02 -7.81070348e-03 -2.08160635e-02 -9.79983062e-02
 -6.89251572e-02  3.06835

In [19]:
print(len(embeddings[0]))

384


In [20]:
sentences = [
    "I love programming",
    "I enjoy coding",
    "The weather is very rainy"
]

sentence_embeddings = model.encode(sentences)

print(sentence_embeddings.shape)

from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(sentence_embeddings)

print(similarity)

(3, 384)
[[ 0.9999999   0.8172092  -0.03298955]
 [ 0.8172092   1.0000001  -0.02056783]
 [-0.03298955 -0.02056783  1.        ]]


In [21]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)
print("First embedding:")
print(embeddings[0])
print("Dimensions:", len(embeddings[0]))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 3
Embedding shape: (3, 384)
First embedding:
[-3.50983553e-02  2.05549262e-02 -2.90550496e-02  1.25653464e-02
 -4.01899070e-02 -1.46645188e-01  5.25082182e-03  5.03757782e-02
 -4.63780686e-02 -2.02275719e-02 -3.43793966e-02  5.00154234e-02
  9.47559848e-02  1.24035478e-02  6.22891001e-02 -6.15294836e-02
 -7.65075311e-02 -1.99534725e-02  1.52971654e-03 -9.58153382e-02
 -5.93157392e-03  6.29708990e-02 -2.34328490e-02  2.24365555e-02
  2.57427823e-02 -1.16095776e-02 -1.07536148e-02 -9.67905112e-03
  2.45132558e-02  6.16728002e-03 -4.83584590e-02  1.01410627e-01
  4.12556641e-02  2.42749825e-02 -4.60887933e-03  5.20044193e-02
 -1.28395809e-02 -8.02411363e-02 -4.75864299e-02  5.49184391e-03
 -3.87001038e-02  3.87854427e-02 -6.10789694e-02 -3.17566171e-02
 -6.91966191e-02  5.25937825e-02 -3.75931971e-02 -1.86671950e-02
 -2.36246437e-02 -1.78272780e-02 -8.83034393e-02  1.01716593e-02
 -5.42289130e-02 -6.09493963e-02 -6.18806621e-03 -1.19047062e-02
  7.03026429e-02 -7.8107034

In [22]:
import faiss
import numpy as np

In [23]:
embeddings = np.array(embeddings).astype("float32")
print(embeddings.shape)
print(embeddings.dtype)

(3, 384)
float32


In [24]:
#find vector dimension
dimension = embeddings.shape[1]
print(dimension)

384


In [26]:
#craete faiss index
index= faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("Number of vectors",index.ntotal)

Number of vectors 3


In [43]:
query="What is rag?"
query_embedding =  model.encode([query]).astype("float32")
distances, indices = index.search(query_embedding, k=2)


In [44]:
for i in indices[0]:
    print(chunks[i])

It is widely used for software development,
Python is a programming language.


#                RAG
                     │
          ┌──────────┴──────────┐
          ↓                     ↓
     RETRIEVAL              GENERATION
          ↓                     ↓
     Embeddings                 LLM
          ↓                     ↑
       FAISS                    │
          ↓                     │
   Relevant chunks              │
          │                     │
          └──────→ Prompt ──────┘
                     ↓
                  Answer

here we are combining retreival information with language generation

In [1]:
import os

print(os.getcwd())
print(os.listdir())

e:\LLM_Sep2026\RAG_Project
['documents', 'steps.ipynb']


In [2]:
import os

print(os.listdir("documents"))

['ml.txt', 'python.txt', 'rag.txt']


In [3]:
files = ["ml.txt", "python.txt", "rag.txt"]

documents = []

for file in files:
    path = os.path.join("documents", file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "source": file,
        "text": text
    })

print("Number of documents:", len(documents))

Number of documents: 3


In [4]:
for document in documents:
    print("SOURCE:", document["source"])
    print(document["text"])
    print("-" * 50)

SOURCE: ml.txt
Machine learning is a branch of artificial intelligence.
It enables computers to learn patterns from data and make
predictions or decisions without being explicitly programmed
for every individual task.
--------------------------------------------------
SOURCE: python.txt
Python is a high-level programming language.
It is widely used for software development, data analysis,
machine learning, automation, and artificial intelligence.
--------------------------------------------------
SOURCE: rag.txt
Retrieval-Augmented Generation combines information retrieval
with language generation. A RAG system retrieves relevant
information from a knowledge base and provides that information
to a language model as context before generating an answer.
--------------------------------------------------


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Create the text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

# Store all chunks along with their source file
all_chunks = []
for document in documents:
    chunks = splitter.split_text(document["text"])

    for chunk in chunks:
        all_chunks.append({
            "source": document["source"],
            "text": chunk
        })

# Display results
print("Total chunks:", len(all_chunks))
for i, chunk in enumerate(all_chunks):
    print(f"\nCHUNK {i}")
    print("SOURCE:", chunk["source"])
    print("TEXT:", chunk["text"])
    print("-" * 60)

Total chunks: 3

CHUNK 0
SOURCE: ml.txt
TEXT: Machine learning is a branch of artificial intelligence.
It enables computers to learn patterns from data and make
predictions or decisions without being explicitly programmed
for every individual task.
------------------------------------------------------------

CHUNK 1
SOURCE: python.txt
TEXT: Python is a high-level programming language.
It is widely used for software development, data analysis,
machine learning, automation, and artificial intelligence.
------------------------------------------------------------

CHUNK 2
SOURCE: rag.txt
TEXT: Retrieval-Augmented Generation combines information retrieval
with language generation. A RAG system retrieves relevant
information from a knowledge base and provides that information
to a language model as context before generating an answer.
------------------------------------------------------------


In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract only the text from each chunk
texts = [chunk["text"] for chunk in all_chunks]

# Convert each chunk into a vector
embeddings = model.encode(texts)

# Convert to NumPy float32 for FAISS
embeddings = np.array(embeddings).astype("float32")

print("Number of chunks:", len(texts))
print("Embedding shape:", embeddings.shape)
print("Embedding dimension:", embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 3
Embedding shape: (3, 384)
Embedding dimension: 384


In [7]:
import faiss

# Get the vector dimension
dimension = embeddings.shape[1]

# Create FAISS index using L2 distance
index = faiss.IndexFlatL2(dimension)

# Add all document embeddings to FAISS
index.add(embeddings)

print("Vector dimension:", dimension)
print("Number of vectors in FAISS:", index.ntotal)

Vector dimension: 384
Number of vectors in FAISS: 3


In [8]:
def retrieve_chunks(query, k=2):

    # Convert query into embedding
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    # Search FAISS
    distances, indices = index.search(query_embedding, k)

    # Store retrieved chunks
    retrieved_chunks = []

    for idx, distance in zip(indices[0], distances[0]):
        retrieved_chunks.append({
            "source": all_chunks[idx]["source"],
            "text": all_chunks[idx]["text"],
            "distance": float(distance)
        })

    return retrieved_chunks


# Test the retrieval function
query = "What is machine learning?"

results = retrieve_chunks(query, k=2)

print("QUERY:", query)
print("=" * 60)

for i, result in enumerate(results, start=1):
    print(f"\nRESULT {i}")
    print("Source:", result["source"])
    print("Distance:", result["distance"])
    print("Text:", result["text"])
    print("-" * 60)

QUERY: What is machine learning?

RESULT 1
Source: ml.txt
Distance: 0.2839607894420624
Text: Machine learning is a branch of artificial intelligence.
It enables computers to learn patterns from data and make
predictions or decisions without being explicitly programmed
for every individual task.
------------------------------------------------------------

RESULT 2
Source: python.txt
Distance: 1.4110718965530396
Text: Python is a high-level programming language.
It is widely used for software development, data analysis,
machine learning, automation, and artificial intelligence.
------------------------------------------------------------


In [10]:
retrieve_chunks("What is Python?")

[{'source': 'python.txt',
  'text': 'Python is a high-level programming language.\nIt is widely used for software development, data analysis,\nmachine learning, automation, and artificial intelligence.',
  'distance': 0.30381524562835693},
 {'source': 'ml.txt',
  'text': 'Machine learning is a branch of artificial intelligence.\nIt enables computers to learn patterns from data and make\npredictions or decisions without being explicitly programmed\nfor every individual task.',
  'distance': 1.4688780307769775}]

In [11]:
retrieve_chunks("What is RAG?")

[{'source': 'rag.txt',
  'text': 'Retrieval-Augmented Generation combines information retrieval\nwith language generation. A RAG system retrieves relevant\ninformation from a knowledge base and provides that information\nto a language model as context before generating an answer.',
  'distance': 1.2063772678375244},
 {'source': 'python.txt',
  'text': 'Python is a high-level programming language.\nIt is widely used for software development, data analysis,\nmachine learning, automation, and artificial intelligence.',
  'distance': 1.8905869722366333}]

In [12]:
retrieve_chunks("How does machine learning work?")

[{'source': 'ml.txt',
  'text': 'Machine learning is a branch of artificial intelligence.\nIt enables computers to learn patterns from data and make\npredictions or decisions without being explicitly programmed\nfor every individual task.',
  'distance': 0.5349899530410767},
 {'source': 'rag.txt',
  'text': 'Retrieval-Augmented Generation combines information retrieval\nwith language generation. A RAG system retrieves relevant\ninformation from a knowledge base and provides that information\nto a language model as context before generating an answer.',
  'distance': 1.5336447954177856}]

In [13]:
pip install -U transformers torch

Note: you may need to restart the kernel to use updated packages.


In [16]:
import os
import getpass
import requests

# Enter your OpenRouter API key securely
# The key will NOT be displayed while you type/paste it.
os.environ["OPENROUTER_API_KEY"] = getpass.getpass(
    "Enter your OpenRouter API key: "
)

# Check that the key was entered
if os.environ.get("OPENROUTER_API_KEY"):
    print("OpenRouter API key configured successfully!")
else:
    print("API key was not entered.")

OpenRouter API key configured successfully!


In [19]:
def generate_answer(query, k=2):

    # --------------------------------------------------
    # 1. Retrieve relevant chunks from FAISS
    # --------------------------------------------------
    retrieved_chunks = retrieve_chunks(query, k=k)

    # --------------------------------------------------
    # 2. Build context from retrieved chunks
    # --------------------------------------------------
    context = "\n\n".join(
        result["text"]
        for result in retrieved_chunks
    )

    # --------------------------------------------------
    # 3. Create the RAG prompt
    # --------------------------------------------------
    prompt = f"""
You are a helpful question-answering assistant.

Answer the question using ONLY the information
provided in the context.

Context:
{context}

Question:
{query}

If the answer cannot be found in the context,
say:
"I don't know based on the provided information."

Give a concise and accurate answer.
"""

    # --------------------------------------------------
    # 4. Send request to OpenRouter
    # --------------------------------------------------
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
            "Content-Type": "application/json"
        },
        json={
            "model": "openrouter/free",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.2,
            "max_tokens": 200
        }
    )

    # --------------------------------------------------
    # 5. Check for API errors
    # --------------------------------------------------
    if response.status_code != 200:
        print("API Error:")
        print(response.text)
        return None

    # --------------------------------------------------
    # 6. Extract the generated answer
    # --------------------------------------------------
    response_data = response.json()

    answer = response_data["choices"][0]["message"]["content"]

    # --------------------------------------------------
    # 7. Return answer + sources
    # --------------------------------------------------
    return {
        "answer": answer,
        "sources": retrieved_chunks
    }

In [22]:
# ======================================================
# TEST THE COMPLETE RAG SYSTEM
# ======================================================

query = input("Enter a query?")

result = generate_answer(query, k=2)

if result:

    print("QUESTION:")
    print(query)

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")

    for source in result["sources"]:
        print("-", source["source"])

QUESTION:
How does RAG use a language model?

ANSWER:
Here, the user is asking me to answer a question based on the provided context. I need to check the context to see if I can find the answer. If not, I should say "I don't know based on the provided information."

Let me read the context carefully:

Context:
"Retrieval-Augmented Generation combines information retrieval
with language generation. A RAG system retrieves relevant
information from a knowledge base and provides that information
to a language model as context before generating an answer.

Python is a high-level programming language.
It is widely used for software development, data analysis,
machine learning, automation, and artificial intelligence."

Question:
"How does RAG use a language model?"

I need to see if the context explains how RAG uses a language model. The context says: "Retrieval-Augmented Generation combines information retrieval with language generation. A RAG system retrieves relevant information from a kn

In [25]:
def generate_answer(query, k=2):

    # --------------------------------------------------
    # 1. Retrieve relevant chunks
    # --------------------------------------------------
    retrieved_chunks = retrieve_chunks(query, k=k)

    # --------------------------------------------------
    # 2. Build context
    # --------------------------------------------------
    context = "\n\n".join(
        result["text"]
        for result in retrieved_chunks
    )

    # --------------------------------------------------
    # 3. Create RAG prompt
    # --------------------------------------------------
    prompt = f"""You are a question-answering assistant.

Use ONLY the information in the context to answer the question.

Context:
{context}

Question:
{query}

Give only the final answer.
Do not show your reasoning.
Do not repeat the context.

Answer:"""

    # --------------------------------------------------
    # 4. Send request to OpenRouter
    # --------------------------------------------------
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
            "Content-Type": "application/json"
        },
        json={
            "model": "openrouter/free",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.0,
            "max_tokens": 150
        }
    )

    # --------------------------------------------------
    # 5. Check HTTP response
    # --------------------------------------------------
    if response.status_code != 200:
        print("API Error:")
        print(response.text)
        return None

    # --------------------------------------------------
    # 6. Read API response
    # --------------------------------------------------
    data = response.json()

    # Debugging: show response if content is missing
    if not data.get("choices"):
        print("Unexpected API response:")
        print(data)
        return None

    message = data["choices"][0].get("message", {})

    answer = message.get("content")

    # --------------------------------------------------
    # 7. Handle empty response
    # --------------------------------------------------
    if not answer:
        print("The model returned an empty answer.")
        print("\nFull API response:")
        print(data)
        return None

    answer = answer.strip()

    # --------------------------------------------------
    # 8. Return answer + sources
    # --------------------------------------------------
    return {
        "answer": answer,
        "sources": retrieved_chunks
    }

In [26]:
# ======================================================
# Interactive user query
# ======================================================

query = input("Enter your question: ")

result = generate_answer(query, k=2)

if result:

    print("\nANSWER:")
    print(result["answer"])

    print("\nSOURCES:")

    for source in result["sources"]:
        print("-", source["source"])


ANSWER:
RAG provides retrieved relevant information to the language model as context, then the language model generates an answer using that context.

SOURCES:
- rag.txt
- python.txt
